In [1]:
import torch
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.1f} GB")

# Простой тест скорости
a = torch.randn(10000, 10000).cuda()
b = torch.randn(10000, 10000).cuda()

start = time.time()
c = torch.matmul(a, b)
torch.cuda.synchronize()
end = time.time()

print(f"Матричное умножение 10000x10000: {(end-start):.2f} сек")
# Должно быть 0.5-1 секунда

PyTorch version: 2.7.0+cu126
CUDA available: True
GPU name: Tesla V100-SXM2-32GB
VRAM: 31.7 GB
Матричное умножение 10000x10000: 0.18 сек


In [2]:
from enum import Enum


class Priority(Enum):
    LOW = 0
    MEDIUM = 1
    HIGH = 2
    CRITICAL = 3


class Category(Enum):
    PAYMENT = 0
    DELIVERY = 1
    TECH = 2
    PRODUCT = 3
    SPAM = 4


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

print("Библиотеки загружены! Готовы обрабатывать тексты.")

Библиотеки загружены! Готовы обрабатывать тексты.


In [4]:
df = pd.read_csv("FullDataset3.csv")

In [5]:
df

,text,category_value,priority_value
0,"Добрый вечер, я столкнулся с проблемой при опл...",0,0
1,"Приветствую, у меня возникла странная ситуация...",0,0
2,"Приветствую, при попытке оплаты возникает ошиб...",0,0
3,"Приветствую, система списала деньги, но заказ ...",0,0
4,"Здравствуйте, я столкнулся с проблемой при опл...",0,0
...,...,...,...
5854,"Здравствуйте! Подскажите, совместим ли данный ...",3,1
5855,Здравствуйте. Я хочу оформить возврат сложной ...,3,2
5856,ВНИМАНИЕ! Ваша банковская карта была выбрана д...,4,0
5857,Здравствуйте! Мы представляем сервис автоматиз...,4,0


In [6]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['category_value'], test_size=0.25, random_state=42)

vectorizer_category = TfidfVectorizer()

X_train_tfidf = vectorizer_category.fit_transform(X_train)

X_test_tfidf = vectorizer_category.transform(X_test)

model_category = LogisticRegression()
model_category.fit(X_train_tfidf, y_train)

y_pred = model_category.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.92      0.94       302
           1       0.94      0.92      0.93       290
           2       0.90      0.92      0.91       295
           3       0.91      0.95      0.93       279
           4       0.96      0.96      0.96       299

    accuracy                           0.93      1465
   macro avg       0.93      0.93      0.93      1465
weighted avg       0.93      0.93      0.93      1465



In [7]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['priority_value'], test_size=0.25, random_state=42)

vectorizer_priority = TfidfVectorizer()

X_train_tfidf = vectorizer_priority.fit_transform(X_train)

X_test_tfidf = vectorizer_priority.transform(X_test)

model_priority = LogisticRegression()
model_priority.fit(X_train_tfidf, y_train)

y_pred = model_priority.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.57      0.62       359
           1       0.58      0.57      0.57       357
           2       0.51      0.60      0.55       377
           3       0.59      0.60      0.60       372

    accuracy                           0.58      1465
   macro avg       0.59      0.58      0.59      1465
weighted avg       0.59      0.58      0.59      1465



In [8]:
y_pred = model_priority.predict(X_test_tfidf)
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix (formatted):")
for i, row in enumerate(cm):
    print(f"True {Priority(i).name}\t: {row}")


Confusion Matrix (formatted):
True LOW	: [206  45  54  54]
True MEDIUM	: [ 30 202  81  44]
True HIGH	: [ 38  59 225  55]
True CRITICAL	: [ 31  41  77 223]


In [13]:
abs(y_pred - y_test).mean()

np.float64(0.6525597269624573)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from statistics import mean

def print_metrics(results):
    print("Loss:", results["loss"])

    print("\n=== Category ===")
    print("Accuracy:", results["category_accuracy"])
    print(classification_report(
        results["cat_labels"],
        results["cat_preds"]
    ))

    print("\n=== Priority ===")
    print("Mean dist:", mean(map(lambda x: abs(x[0] - x[1]), zip(results["pr_labels"], results['pr_preds']))))
    print("Accuracy:", results["priority_accuracy"])
    print(classification_report(
        results["pr_labels"],
        results["pr_preds"]
    ))

    # Confusion Matrix
    cm = confusion_matrix(results["pr_labels"], results["pr_preds"])

    # более читаемый вид
    print("\nConfusion Matrix (formatted):")
    for i, row in enumerate(cm):
        print(f"True {Priority(i).name}\t: {row}")

In [ ]:
evaluate_model

In [9]:
user_input = "Оплата не прошла"
user_cttfidf = vectorizer_category.transform([user_input])
user_prtfidf = vectorizer_priority.transform([user_input])
print(Category(model_category.predict(user_cttfidf)), Priority(model_priority.predict(user_prtfidf)))

Category.PAYMENT Priority.HIGH
